<a href="https://colab.research.google.com/github/Midhula13/CSA6506-GEN-AI/blob/main/Gen_Ai_21to32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the SDKs used in this notebook.
!pip -q install -U openai google-genai huggingface_hub pandas tabulate sqlglot

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.2/792.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 18.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires g

In [ ]:
import os, re, json, textwrap, ast, sqlite3
from getpass import getpass
import pandas as pd
from IPython.display import display, Markdown

# Securely read Colab Secrets. No key is printed.
def get_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return os.environ.get(name)

GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

# Change these model IDs if your provider account uses different models.
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gpt-4o-mini")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")
HF_MODEL = os.environ.get("HF_MODEL", "HuggingFaceH4/zephyr-7b-beta")

print("Keys available:", {
    "OpenAI": bool(GEMINI_API_KEY),
    "Gemini": bool(GEMINI_API_KEY),
    "Hugging Face": bool(HF_TOKEN),
})

# One Gemini function used by all 12 experiments.
from google import genai

def gemini_generate(prompt, model=GEMINI_MODEL):
    if not GEMINI_API_KEY:
        return "[Skipped: add GEMINI_API_KEY to Colab Secrets.]"
    client = genai.Client(api_key=GEMINI_API_KEY)
    response = client.models.generate_content(model=model, contents=prompt)
    return response.text


Keys available: {'OpenAI': True, 'Gemini': True, 'Hugging Face': False}


## 1. Smart Fitness Watch product description

In [ ]:
product_task = """
Write a product description for a Smart Fitness Watch.
Audience: health-conscious adults aged 20-45.
Mention: heart-rate monitoring, sleep tracking, step counting, workout modes,
water resistance, smartphone notifications, battery life, and privacy.
Tone: persuasive but accurate. Length: 120-150 words. Use a headline and one paragraph.
Do not invent a brand name, medical diagnosis capability, or unsupported numerical claims.
"""

product_zero_shot = product_task

product_one_shot = """
Example:
Task: Write a product description for wireless earbuds. Tone: concise and persuasive.
Output: SoundFlow wireless earbuds deliver clear audio, comfortable all-day wear, and
reliable touch controls in a compact design. With a long-lasting case and secure fit,
they make calls, music, and podcasts easier wherever your day takes you.

New task:
""" + product_task

product_few_shot = """
Example 1:
Task: Product description for a reusable water bottle.
Output: Stay refreshed with a durable, leak-resistant bottle designed for commutes,
workouts, and weekend adventures. Its easy-clean design and insulated construction
help keep hydration convenient throughout the day.

Example 2:
Task: Product description for a fitness mat.
Output: Build a steadier routine with a supportive, non-slip fitness mat made for
stretching, yoga, and home workouts. Its cushioned surface helps improve comfort while
its lightweight design makes storage and travel simple.

New task:
""" + product_task

print("ZERO-SHOT PROMPT:\n", product_zero_shot)
print("\nONE-SHOT PROMPT:\n", product_one_shot)
print("\nFEW-SHOT PROMPT:\n", product_few_shot)

# Generate all three versions with Gemini using the active 'gemini-3.8-flash' model.
for label, prompt in {"Zero-shot": product_zero_shot, "One-shot": product_one_shot, "Few-shot": product_few_shot}.items():
    try:
        output = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- {label} Gemini output ---\n{output}")
    except Exception as e:
        print(f"\n--- {label} Gemini output failed due to temporary service issue: {e} ---")

ZERO-SHOT PROMPT:
 
Write a product description for a Smart Fitness Watch.
Audience: health-conscious adults aged 20-45.
Mention: heart-rate monitoring, sleep tracking, step counting, workout modes,
water resistance, smartphone notifications, battery life, and privacy.
Tone: persuasive but accurate. Length: 120-150 words. Use a headline and one paragraph.
Do not invent a brand name, medical diagnosis capability, or unsupported numerical claims.


ONE-SHOT PROMPT:
 
Example:
Task: Write a product description for wireless earbuds. Tone: concise and persuasive.
Output: SoundFlow wireless earbuds deliver clear audio, comfortable all-day wear, and
reliable touch controls in a compact design. With a long-lasting case and secure fit,
they make calls, music, and podcasts easier wherever your day takes you.

New task:

Write a product description for a Smart Fitness Watch.
Audience: health-conscious adults aged 20-45.
Mention: heart-rate monitoring, sleep tracking, step counting, workout modes,

## 2. 200-word blog: Applications of Artificial Intelligence in Healthcare

In [ ]:
blog_task = """
Write a blog of exactly 200 words titled 'Applications of Artificial Intelligence in Healthcare'.
Explain at least five applications, including medical imaging, drug discovery,
personalized treatment, patient monitoring, and administrative automation.
Use clear language for a general audience. Mention one benefit and one limitation or ethical concern.
Return only the title and blog body. Check the word count before responding.
"""

blog_zero_shot = blog_task

blog_one_shot = """
Example:
Task: Write a 100-word blog about renewable energy for a general audience. Include a title,
three applications, one benefit, and one limitation.
Output:
Renewable Energy in Everyday Life
Renewable energy is moving from distant power stations into daily life. Solar panels can
reduce household electricity costs, while wind farms supply clean power to communities.
Hydroelectric systems also generate electricity by using flowing water. These technologies
can reduce greenhouse-gas emissions and diversify energy supplies. However, renewable
projects may require high initial investment and careful land-use planning. Their output can
also vary with sunlight, wind, or rainfall. Continued improvements in batteries, grids, and
energy management are making clean power more dependable. As costs fall and technology
improves, renewable energy can support a healthier environment while meeting growing demand.

New task:
""" + blog_task

blog_few_shot = """
Example 1:
Task: Explain three uses of AI in education in 80 words, with a title and balanced tone.
Output: AI in Education: Practical Uses
AI can personalize practice by adapting questions to a learner's progress. It can provide
quick feedback on writing and help teachers identify topics that need review. It can also
support accessibility through speech, translation, and reading tools. Benefits include
scalable support and faster feedback, but privacy, bias, and overreliance require careful
human oversight.

Example 2:
Task: Explain three uses of robotics in manufacturing in 80 words, with a title and balanced tone.
Output: Robotics in Modern Manufacturing
Robots can assemble products with consistent precision, inspect items for defects, and move
materials through warehouses. These uses can improve safety and productivity, especially for
repetitive or hazardous work. At the same time, organizations must invest in training,
maintenance, and responsible workforce transitions. Human judgment remains important for
exceptional cases and quality decisions.

New task:
""" + blog_task

print(blog_zero_shot, "\n---\n", blog_one_shot, "\n---\n", blog_few_shot)

# Generate all three versions with Gemini using the active 'gemini-3.8-flash' model.
for label, prompt in {"Zero-shot": blog_zero_shot, "One-shot": blog_one_shot, "Few-shot": blog_few_shot}.items():
    try:
        output = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- {label} Gemini output ---\n{output}")
    except Exception as e:
        print(f"\n--- {label} Gemini output failed due to temporary service issue: {e} ---")


Write a blog of exactly 200 words titled 'Applications of Artificial Intelligence in Healthcare'.
Explain at least five applications, including medical imaging, drug discovery,
personalized treatment, patient monitoring, and administrative automation.
Use clear language for a general audience. Mention one benefit and one limitation or ethical concern.
Return only the title and blog body. Check the word count before responding.
 
---
 
Example:
Task: Write a 100-word blog about renewable energy for a general audience. Include a title,
three applications, one benefit, and one limitation.
Output:
Renewable Energy in Everyday Life
Renewable energy is moving from distant power stations into daily life. Solar panels can
reduce household electricity costs, while wind farms supply clean power to communities.
Hydroelectric systems also generate electricity by using flowing water. These technologies
can reduce greenhouse-gas emissions and diversify energy supplies. However, renewable
projects m

## 3. Summarize an article into exactly 50 words

Replace the sample article below with the article given in your assignment. The evaluation
function checks approximate word count and compares the summary against the article using
simple coverage signals. For a stronger evaluation, use the LLM evaluator in Task 12.

In [ ]:
ARTICLE_TEXT = """
Artificial intelligence is increasingly used in hospitals to support diagnosis, treatment,
research, and administration. Machine-learning systems can identify patterns in medical
images, predict patient risks, and help researchers discover drug candidates. Remote-monitoring
systems can alert clinicians when a patient's measurements change. These tools may improve
speed and access, but they require representative data, privacy protections, clinical validation,
and human oversight. AI should support healthcare professionals rather than replace their
judgment, especially in high-stakes decisions.
""".strip()

summary_task = f"""
Summarize the article below in exactly 50 words. Preserve the central ideas, important benefits,
and key limitations. Do not add facts that are absent from the article. Return only the summary.

ARTICLE:
{ARTICLE_TEXT}
"""
summary_one_shot = """
Example article: Solar energy converts sunlight into electricity and can reduce emissions.
However, production depends on weather and requires storage or grid support.
Example 50-word summary: Solar power transforms sunlight into electricity, helping reduce
emissions and diversify energy supplies. Its usefulness is growing as panels become cheaper
and more efficient. Yet generation varies with weather, so dependable adoption requires
storage, flexible grids, careful planning, and consideration of manufacturing impacts and land use.

New article to summarize in exactly 50 words:
""" + ARTICLE_TEXT
summary_few_shot = """
Example 1: A library extended evening hours, giving students more study time. Attendance rose,
but staffing costs increased. The library plans to evaluate usage before making the schedule permanent.
Summary: Extending library hours increased student access and attendance, but also raised staffing
costs. The library will review usage data before deciding whether to keep the new schedule permanently.

Example 2: Telemedicine connects patients with clinicians remotely, improving access for people far
from hospitals. It may reduce travel, but poor connectivity and limited physical examination can restrict care.
Summary: Telemedicine expands remote access and reduces travel, especially for distant patients, but
connectivity problems and the inability to perform complete physical examinations can limit its usefulness.

New article to summarize in exactly 50 words:
""" + ARTICLE_TEXT

print(summary_task)
print("\nONE-SHOT:\n", summary_one_shot)
print("\nFEW-SHOT:\n", summary_few_shot)

# Generate all three 50-word summaries with Gemini using the active 'gemini-3.8-flash' model.
summary_outputs = {}
for label, prompt in {"Zero-shot": summary_task, "One-shot": summary_one_shot, "Few-shot": summary_few_shot}.items():
    try:
        summary_outputs[label] = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- {label} Gemini summary ---\n{summary_outputs[label]}")
    except Exception as e:
        summary_outputs[label] = f"[Failed due to service issue: {e}]"
        print(f"\n--- {label} Gemini summary failed due to temporary service issue: {e} ---")


Summarize the article below in exactly 50 words. Preserve the central ideas, important benefits,
and key limitations. Do not add facts that are absent from the article. Return only the summary.

ARTICLE:
Artificial intelligence is increasingly used in hospitals to support diagnosis, treatment,
research, and administration. Machine-learning systems can identify patterns in medical
images, predict patient risks, and help researchers discover drug candidates. Remote-monitoring
systems can alert clinicians when a patient's measurements change. These tools may improve
speed and access, but they require representative data, privacy protections, clinical validation,
and human oversight. AI should support healthcare professionals rather than replace their
judgment, especially in high-stakes decisions.


ONE-SHOT:
 
Example article: Solar energy converts sunlight into electricity and can reduce emissions.
However, production depends on weather and requires storage or grid support.
Example 50-w

### 3A. Summary comparison rubric

Score each output from 1 to 5:

- **Accuracy:** Does it preserve the article's facts without hallucination?
- **Completeness:** Does it include the central topic, benefits, limitations, and conclusion?
- **Readability:** Is it clear, grammatical, concise, and close to 50 words?

In [ ]:
summary_comparison_template = pd.DataFrame([
    {"Prompt": "Zero-shot", "Accuracy (1-5)": None, "Completeness (1-5)": None, "Readability (1-5)": None, "Notes": ""},
    {"Prompt": "One-shot", "Accuracy (1-5)": None, "Completeness (1-5)": None, "Readability (1-5)": None, "Notes": ""},
    {"Prompt": "Few-shot", "Accuracy (1-5)": None, "Completeness (1-5)": None, "Readability (1-5)": None, "Notes": ""},
])
display(summary_comparison_template)

,Prompt,Accuracy (1-5),Completeness (1-5),Readability (1-5),Notes
0,Zero-shot,None,None,None,
1,One-shot,None,None,None,
2,Few-shot,None,None,None,


## 4. Professional illness-leave email

In [ ]:
email_task = """
Write a professional email requesting leave due to illness.
Use placeholders [Manager Name], [Your Name], [start date], [end date], and [urgent handover details].
Include a clear subject line, polite request, brief reason without unnecessary medical details,
leave dates, availability expectations, handover plan, and a professional closing.
Return a properly formatted email.
"""
email_zero_shot = email_task
email_one_shot = """
Example:
Task: Write a professional email requesting one day of leave for a family appointment.
Output:
Subject: Leave Request for [date]

Dear [Manager Name],

I am writing to request leave on [date] due to an important family appointment. I will
complete [priority task] before leaving and share any necessary handover notes with [colleague].
I will remain reachable for urgent matters if needed.

Thank you for your consideration.

Kind regards,
[Your Name]

New task:
""" + email_task
email_few_shot = """
Example 1:
Task: Request leave for a personal appointment.
Output: Subject: Leave Request for [date]

Dear [Manager Name],

I would like to request leave on [date] for a personal appointment. I will finish my urgent work beforehand and provide a handover note. Thank you for considering my request.

Kind regards,
[Your Name]

Example 2:
Task: Notify a manager about two days of illness leave.
Output: Subject: Sick Leave Notification: [start date] to [end date]

Dear [Manager Name],

I am unwell and need to take sick leave from [start date] through [end date]. I will update you if my recovery timeline changes and will share [urgent handover details] so that current priorities remain covered.

Sincerely,
[Your Name]

New task:
""" + email_task
print(email_zero_shot, "\n---\n", email_one_shot, "\n---\n", email_few_shot)

# Generate all three emails with Gemini using the active 'gemini-3.8-flash' model.
for label, prompt in {"Zero-shot": email_zero_shot, "One-shot": email_one_shot, "Few-shot": email_few_shot}.items():
    try:
        output = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- {label} Gemini email ---\n{output}")
    except Exception as e:
        print(f"\n--- {label} Gemini email failed due to temporary service issue: {e} ---")


Write a professional email requesting leave due to illness.
Use placeholders [Manager Name], [Your Name], [start date], [end date], and [urgent handover details].
Include a clear subject line, polite request, brief reason without unnecessary medical details,
leave dates, availability expectations, handover plan, and a professional closing.
Return a properly formatted email.
 
---
 
Example:
Task: Write a professional email requesting one day of leave for a family appointment.
Output:
Subject: Leave Request for [date]

Dear [Manager Name],

I am writing to request leave on [date] due to an important family appointment. I will
complete [priority task] before leaving and share any necessary handover notes with [colleague].
I will remain reachable for urgent matters if needed.

Thank you for your consideration.

Kind regards,
[Your Name]

New task:

Write a professional email requesting leave due to illness.
Use placeholders [Manager Name], [Your Name], [start date], [end date], and [urge

### 4A. Email comparison rubric

Assess **tone**, **grammar**, **formatting**, and **completeness** from 1 to 5. A strong email is polite,
clear, appropriately brief, formatted with a subject and greeting, and includes dates plus a handover plan.

In [ ]:
email_comparison_template = pd.DataFrame([
    {"Prompt": "Zero-shot", "Tone": None, "Grammar": None, "Formatting": None, "Completeness": None, "Notes": ""},
    {"Prompt": "One-shot", "Tone": None, "Grammar": None, "Formatting": None, "Completeness": None, "Notes": ""},
    {"Prompt": "Few-shot", "Tone": None, "Grammar": None, "Formatting": None, "Completeness": None, "Notes": ""},
])
display(email_comparison_template)

,Prompt,Tone,Grammar,Formatting,Completeness,Notes
0,Zero-shot,None,None,None,None,
1,One-shot,None,None,None,None,
2,Few-shot,None,None,None,None,


## 5. Promotional social-media post for an AI Workshop

In [ ]:
social_task = """
Create a promotional social-media post for an AI Workshop.
Audience: university students and early-career professionals.
Include an attention-grabbing opening, workshop benefits, beginner-friendly positioning,
call to action, and placeholders [date], [time], [venue/link], and [registration link].
Keep it under 100 words and add 4-6 relevant hashtags. Do not promise guaranteed jobs or income.
"""
social_zero_shot = social_task
social_one_shot = """
Example:
Task: Promote a beginner Excel workshop in under 60 words with a call to action and hashtags.
Output: Turn spreadsheets into smart decisions. Join our beginner-friendly Excel Workshop to
learn formulas, charts, and practical data tips through hands-on examples. Save your seat for
[date] at [venue/link]. Register: [registration link]. #Excel #DataSkills #Workshop

New task:
""" + social_task
social_few_shot = """
Example 1:
Task: Promote a public-speaking workshop for students.
Output: Find your voice and speak with confidence. Our practical workshop helps students plan,
practice, and deliver stronger presentations in a supportive setting. Join us [date] at [venue/link].
Register: [registration link]. #PublicSpeaking #Students #SkillBuilding

Example 2:
Task: Promote a coding workshop for beginners.
Output: Curious about coding? Start here. Build your first small program, ask questions, and
learn with friendly guidance—no previous experience required. Meet us [date] at [venue/link].
Register: [registration link]. #Coding #Beginners #TechSkills

New task:
""" + social_task
print(social_zero_shot, "\n---\n", social_one_shot, "\n---\n", social_few_shot)

# Generate all three social posts with Gemini using the active 'gemini-3.8-flash' model.
for label, prompt in {"Zero-shot": social_zero_shot, "One-shot": social_one_shot, "Few-shot": social_few_shot}.items():
    try:
        output = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- {label} Gemini post ---\n{output}")
    except Exception as e:
        print(f"\n--- {label} Gemini post failed due to temporary service issue: {e} ---")


Create a promotional social-media post for an AI Workshop.
Audience: university students and early-career professionals.
Include an attention-grabbing opening, workshop benefits, beginner-friendly positioning,
call to action, and placeholders [date], [time], [venue/link], and [registration link].
Keep it under 100 words and add 4-6 relevant hashtags. Do not promise guaranteed jobs or income.
 
---
 
Example:
Task: Promote a beginner Excel workshop in under 60 words with a call to action and hashtags.
Output: Turn spreadsheets into smart decisions. Join our beginner-friendly Excel Workshop to
learn formulas, charts, and practical data tips through hands-on examples. Save your seat for
[date] at [venue/link]. Register: [registration link]. #Excel #DataSkills #Workshop

New task:

Create a promotional social-media post for an AI Workshop.
Audience: university students and early-career professionals.
Include an attention-grabbing opening, workshop benefits, beginner-friendly positioning,


## 6. Text Generation using the Gemini LLM API

This section uses the same `GEMINI_API_KEY` secret for general user-provided prompts.


In [16]:
user_prompt = input("Enter a prompt for Gemini (or press Enter for a demo): " ).strip()
if not user_prompt:
    user_prompt = "Explain prompt engineering in five simple bullet points."
print(gemini_generate(user_prompt))


KeyboardInterrupt: Interrupted by user

## 7. Text Generation using the Google Gemini LLM API

In [14]:
from google import genai

# Fallback to default model if GEMINI_MODEL isn't defined in the global namespace
active_model = globals().get("GEMINI_MODEL", "gemini-2.0-flash")

def gemini_generate(prompt, model=active_model):
    has_key = globals().get("GEMINI_API_KEY", None)
    if not has_key:
        return "[Skipped: add GEMINI_API_KEY to Colab Secrets.]"
    client = genai.Client(api_key=has_key)
    response = client.models.generate_content(model=model, contents=prompt)
    return response.text

gemini_prompt = input("Enter a prompt for Gemini (or press Enter for a demo): ").strip()
if not gemini_prompt:
    gemini_prompt = "Give three practical uses of AI in education and one limitation."
print(gemini_generate(gemini_prompt))


KeyboardInterrupt



## 8. Gemini text generation for multiple user-provided prompts

The original assignment names Hugging Face for this activity. Because you requested one API for all 12 experiments, this adapted version uses your Gemini API key consistently.


In [ ]:
prompts = [
    "Write a concise paragraph about why data privacy matters in AI.",
    "List three applications of AI in education and one limitation.",
    "Explain supervised learning to a beginner in 80 words."
]
for i, prompt in enumerate(prompts, 1):
    try:
        response = gemini_generate(prompt, model='gemini-3.8-flash')
        print(f"\n--- Gemini response {i} ---\n{response}")
    except Exception as e:
        print(f"\n--- Gemini response {i} failed due to API status: {e} ---")


--- Gemini response 1 failed due to API status: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 16h50m15.000821804s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 

## 9. Python code generation using structured prompting

The prompt requests a specification, algorithm, assumptions, complexity, code, and tests.
The notebook then performs basic static validation and executes the generated code in a subprocess.
Never execute untrusted generated code outside a safe educational environment.

In [ ]:
python_problem = "Write a Python function named `is_prime(n)` that returns True if n is prime and False otherwise. Include input validation and five assertions."
structured_python_prompt = f"""
You are a careful Python programmer. Solve this computational problem:
{python_problem}

Return exactly these sections:
1. ASSUMPTIONS
2. ALGORITHM
3. COMPLEXITY
4. CODE - a complete Python code block only
5. TESTS - at least five assertions

Constraints: use Python 3, standard library only, clear names, no network access,
and do not use eval or exec. Ensure the code is syntactically valid.
"""
print(structured_python_prompt)


You are a careful Python programmer. Solve this computational problem:
Write a Python function named `is_prime(n)` that returns True if n is prime and False otherwise. Include input validation and five assertions.

Return exactly these sections:
1. ASSUMPTIONS
2. ALGORITHM
3. COMPLEXITY
4. CODE - a complete Python code block only
5. TESTS - at least five assertions

Constraints: use Python 3, standard library only, clear names, no network access,
and do not use eval or exec. Ensure the code is syntactically valid.



In [ ]:
def extract_code_block(text):
    match = re.search(r"```(?:python)?\s*(.*?)""", text, flags=re.S | re.I)
    return match.group(1).strip() if match else text.strip()

def validate_python_code(code_text):
    report = {"syntax_valid": False, "has_function": False, "uses_dangerous_calls": [], "errors": []}
    try:
        tree = ast.parse(code_text)
        report["syntax_valid"] = True
        report["has_function"] = any(isinstance(n, ast.FunctionDef) for n in ast.walk(tree))
        dangerous = {"eval", "exec", "__import__", "system", "popen"}
        report["uses_dangerous_calls"] = sorted({n.func.id for n in ast.walk(tree) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id in dangerous})
    except SyntaxError as e:
        report["errors"].append(str(e))
    return report

if GEMINI_API_KEY:
    try:
        generated_python = gemini_generate(structured_python_prompt, model='gemini-3.8-flash')
        python_code = extract_code_block(generated_python)
        print(generated_python)
        print("\nVALIDATION:\n", json.dumps(validate_python_code(python_code), indent=2))
    except Exception as e:
        print(f"Code generation failed due to API status: {e}")
else:
    print("Add GEMINI_API_KEY to generate and validate code automatically.")

Code generation failed due to API status: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


## 10. SQL query generation using structured prompting

The schema and requirements are supplied explicitly. The query is checked by parsing it with SQLite
and by verifying that it is read-only. Adjust the dialect note if you need PostgreSQL, MySQL, or another database.

In [12]:
schema = """
customers(customer_id INTEGER PRIMARY KEY, name TEXT, country TEXT)
orders(order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date DATE, total_amount REAL,
       FOREIGN KEY(customer_id) REFERENCES customers(customer_id))
"""
sql_requirements = """
Return the top five customers by total order value during 2025, including customer name,
country, number of orders, and total value. Include customers with no orders only if they
had an order in 2025; sort by total value descending. Use standard SQLite SQL.
"""
structured_sql_prompt = f"""
You are a SQL expert. Database schema:
{schema}

Requirement:
{sql_requirements}

Return exactly:
1. ASSUMPTIONS
2. SQL - one read-only SELECT query in a code block
3. EXPLANATION - explain joins, filters, grouping, and ordering

Do not use INSERT, UPDATE, DELETE, DROP, ALTER, or other data-modifying statements.
"""
print(structured_sql_prompt)


You are a SQL expert. Database schema:

customers(customer_id INTEGER PRIMARY KEY, name TEXT, country TEXT)
orders(order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date DATE, total_amount REAL,
       FOREIGN KEY(customer_id) REFERENCES customers(customer_id))


Requirement:

Return the top five customers by total order value during 2025, including customer name,
country, number of orders, and total value. Include customers with no orders only if they
had an order in 2025; sort by total value descending. Use standard SQLite SQL.


Return exactly:
1. ASSUMPTIONS
2. SQL - one read-only SELECT query in a code block
3. EXPLANATION - explain joins, filters, grouping, and ordering

Do not use INSERT, UPDATE, DELETE, DROP, ALTER, or other data-modifying statements.



In [11]:
def validate_sql(sql_text):
    cleaned = re.sub(r"```sql|```", "", sql_text, flags=re.I).strip()
    first = cleaned.lstrip().split(None, 1)[0].upper() if cleaned else ""
    forbidden = re.findall(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|TRUNCATE|REPLACE)\b", cleaned, flags=re.I)
    return {"starts_with_select_or_with": first in {"SELECT", "WITH"}, "forbidden_keywords": sorted(set(x.upper() for x in forbidden)), "query": cleaned}

has_key = globals().get("GEMINI_API_KEY", None)
if has_key:
    generated_sql = gemini_generate(structured_sql_prompt)
    sql_match = re.search(r"```(?:sql)?\s*(.*?)```", generated_sql, flags=re.S | re.I)
    sql_query = sql_match.group(1).strip() if sql_match else generated_sql.strip()
    print(generated_sql)
    print("\nVALIDATION:\n", json.dumps(validate_sql(sql_query), indent=2))
else:
    print("Add GEMINI_API_KEY to generate and validate SQL automatically.")

Add GEMINI_API_KEY to generate and validate SQL automatically.


## 11. Comparative analysis of zero-shot, one-shot, and few-shot prompting

This reusable function generates the same task three ways and records the outputs. The rubric is designed
for manual scoring or can be sent to an LLM evaluator in Task 12.

In [5]:
import pandas as pd

def compare_prompting(task, one_shot_example, few_shot_examples):
    prompts = {
        "Zero-shot": task,
        "One-shot": f"Example:\n{one_shot_example}\n\nNew task:\n{task}",
        "Few-shot": f"Examples:\n{few_shot_examples}\n\nNew task:\n{task}",
    }
    rows = []
    # Safeguard against NameError if GEMINI_API_KEY is not defined in globals
    has_key = globals().get("GEMINI_API_KEY", None)
    for style, prompt in prompts.items():
        output = gemini_generate(prompt) if has_key else "[Add GEMINI_API_KEY]"
        rows.append({"Prompting style": style, "Output": output})
    return pd.DataFrame(rows)

comparison_df = compare_prompting(
    "Explain why secure API-key storage matters in a Colab notebook in 80 words.",
    "Explain why password managers matter in 40 words. Output: Password managers store credentials securely, generate stronger passwords, and reduce reuse. They help users avoid writing secrets in notebooks or plain text. A trusted manager, protected by a strong master password and multi-factor authentication, lowers exposure while keeping access convenient.",
    "Example 1: Explain why backups matter in 30 words. Output: Backups protect important files from accidental deletion, hardware failure, and ransomware. Regular, tested copies make recovery faster and reduce disruption.\n\nExample 2: Explain why multi-factor authentication matters in 30 words. Output: Multi-factor authentication adds another verification step beyond a password, making account takeover harder even when a password is exposed."
)
display(comparison_df)

,Prompting style,Output
0,Zero-shot,[Add GEMINI_API_KEY]
1,One-shot,[Add GEMINI_API_KEY]
2,Few-shot,[Add GEMINI_API_KEY]


### Suggested comparison table for Task 11

Score each response from 1 to 5 for **quality**, **relevance**, **accuracy**, and **consistency**.
Also record whether it followed the requested word count and format. Few-shot prompting often improves
format consistency when the examples closely match the target task, but poor or inconsistent examples can
also teach the wrong pattern.

In [9]:
task11_rubric = pd.DataFrame([
    {"Prompting style": "Zero-shot", "Quality": None, "Relevance": None, "Accuracy": None, "Consistency": None, "Format followed?": None},
    {"Prompting style": "One-shot", "Quality": None, "Relevance": None, "Accuracy": None, "Consistency": None, "Format followed?": None},
    {"Prompting style": "Few-shot", "Quality": None, "Relevance": None, "Accuracy": None, "Consistency": None, "Format followed?": None},
])
display(task11_rubric)

,Prompting style,Quality,Relevance,Accuracy,Consistency,Format followed?
0,Zero-shot,None,None,None,None,None
1,One-shot,None,None,None,None,None
2,Few-shot,None,None,None,None,None


## 12. Prompt evaluation, comparison, and refinement using LLM responses

The evaluator receives the task, candidate prompt, and generated answer. It returns JSON scores for
relevance, accuracy, completeness, clarity, and format adherence, plus concrete improvement advice.
The refinement step then creates a stronger prompt using the best candidate and evaluator feedback.

In [8]:
import json
import re

def evaluate_output(task, candidate_prompt, output):
    evaluation_prompt = f"""
You are a strict prompt-quality evaluator. Evaluate the answer against the task.

TASK:
{task}

CANDIDATE PROMPT:
{candidate_prompt}

ANSWER:
{output}

Return valid JSON only with this schema:
{{
  "relevance": 1-5,
  "accuracy": 1-5,
  "completeness": 1-5,
  "clarity": 1-5,
  "format_adherence": 1-5,
  "total": 5-25,
  "strengths": ["..."],
  "improvements": ["..."]
}}
"""
    has_key = globals().get("GEMINI_API_KEY", None)
    if not has_key:
        return {"status": "skipped", "reason": "Add GEMINI_API_KEY"}
    raw = gemini_generate(evaluation_prompt)
    match = re.search(r"\{.*\}", raw, flags=re.S)
    try:
        return json.loads(match.group(0)) if match else {"raw": raw}
    except json.JSONDecodeError:
        return {"raw": raw}

def refine_prompt(task, candidate_prompt, evaluation):
    refinement_prompt = f"""
Improve the following prompt for the task below.

TASK:
{task}

CURRENT PROMPT:
{candidate_prompt}

EVALUATION:
{json.dumps(evaluation, indent=2)}

Return only the refined prompt. It must state the role, task, audience, constraints,
quality checks, and exact output format. Do not include the answer itself.
"""
    has_key = globals().get("GEMINI_API_KEY", None)
    return gemini_generate(refinement_prompt) if has_key else "[Add GEMINI_API_KEY]"

refinement_task = "Write a 50-word summary of the supplied article without adding facts."
candidate_prompts = {
    "A": refinement_task,
    "B": refinement_task + " Preserve the central claim and the main evidence. Return only the summary.",
    "C": refinement_task + " Count the words, remove repetition, preserve important limitations, and check every claim against the article before responding. Return exactly 50 words and nothing else.",
}

has_key = globals().get("GEMINI_API_KEY", None)
results = []
for label, prompt in candidate_prompts.items():
    output = gemini_generate(prompt) if has_key else "[Add GEMINI_API_KEY]"
    evaluation = evaluate_output(refinement_task, prompt, output)
    results.append({"Candidate": label, "Prompt": prompt, "Output": output, "Evaluation": evaluation})

if has_key:
    for item in results:
        print(f"\n--- Candidate {item['Candidate']} ---")
        print("Output:", item["Output"])
        print("Evaluation:", json.dumps(item["Evaluation"], indent=2))
    scored = [r for r in results if isinstance(r["Evaluation"], dict) and isinstance(r["Evaluation"].get("total"), (int, float))]
    if scored:
        best = max(scored, key=lambda r: r["Evaluation"]["total"])
        print("\nBEST CANDIDATE:", best["Candidate"])
        print("\nREFINED PROMPT:\n", refine_prompt(refinement_task, best["Prompt"], best["Evaluation"]))
else:
    print("Add GEMINI_API_KEY to run automatic evaluation and refinement.")

Add GEMINI_API_KEY to run automatic evaluation and refinement.
